# Pretrained ST-GCN++ Fine-Tuning for Dance Embedding

이 노트북은 **외부 사전학습 모델**(PYSKL ST-GCN++, NTU-60)을 다운로드하고,
로컬 댄스 reference 데이터로 fine-tuning하여 서비스의 `embedding` 옵션에서
사용할 `.tflite` 모델을 만드는 end-to-end 노트북이다.

기존 `embedding_finetuning.ipynb`(MPOSE2021 pretrain → dance fine-tune)과 같은
큰 구조를 따르되, **사전학습 모델이 외부 ST-GCN++** 이라는 점이 다르다.

### 파이프라인

```
PYSKL ST-GCN++ (.pth, NTU-60, COCO 17 joints)
  → COCO 17 → 12 DANCE_JOINTS 가중치 적응
  → classification head 제거, embedding projection head 추가
  → 댄스 reference 데이터로 triplet fine-tuning (PyTorch)
  → PyTorch → ONNX → TFLite export
  → ScratchPoseSimilarity runtime smoke test
```

### 입력

```text
data/models/pretrain/stgcnpp/stgcnpp_ntu60_xsub_hrnet_j.pth
```

### 출력

```text
data/models/embedding/{name}.tflite
data/models/embedding/{name}.pt
data/models/embedding/{name}_meta.json
```

## 1. 환경과 프로젝트 루트 확인

노트북이 어느 위치에서 열려도 `pjt_main` 루트를 찾아서 이후 셀들이 같은 경로 기준으로 동작하도록 한다.

In [1]:
from pathlib import Path
import json
import subprocess
import sys
import time


def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for path in [start, *start.parents]:
        if (path / 'scripts' / 'finetune_pretrained_stgcn.py').exists():
            return path
    raise RuntimeError('Could not find pjt_main project root')


PROJECT_ROOT = find_project_root()
for import_path in (PROJECT_ROOT, PROJECT_ROOT / 'src'):
    import_path = str(import_path)
    if import_path not in sys.path:
        sys.path.insert(0, import_path)

print('PROJECT_ROOT =', PROJECT_ROOT)
print('PYTHON =', sys.executable)

PROJECT_ROOT = /workspace/users/yijin/boot_env/pjt_main
PYTHON = /workspace/users/yijin/boot_env/.venv/bin/python


## 2. 사전학습 Checkpoint 다운로드 및 확인

PYSKL ST-GCN++는 NTU-RGB+D 60 데이터셋에서 HRNet 2D skeleton (COCO 17 joints)으로 학습된 모델이다.
공식 checkpoint를 다운로드한다.

- 아키텍처: ST-GCN++ (10 블록, 3-subset adaptive adjacency, Multi-Scale TCN)
- 입력: `(N, 3, T, 17)` — 3ch(x,y,conf), T frames, 17 COCO joints
- 출력: 60 action classes
- 총 파라미터: ~530K

In [2]:
CHECKPOINT_DIR = PROJECT_ROOT / 'data' / 'models' / 'pretrain' / 'stgcnpp'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT_URL = 'https://download.openmmlab.com/mmaction/pyskl/ckpt/stgcnpp/stgcnpp_ntu60_xsub_hrnet/j.pth'
CHECKPOINT_PATH = CHECKPOINT_DIR / 'stgcnpp_ntu60_xsub_hrnet_j.pth'

if not CHECKPOINT_PATH.exists():
    print(f'Downloading ST-GCN++ checkpoint...')
    subprocess.check_call([
        'wget', '-q', '--show-progress',
        CHECKPOINT_URL, '-O', str(CHECKPOINT_PATH),
    ])
    print(f'Downloaded: {CHECKPOINT_PATH}')
else:
    print(f'Checkpoint already exists: {CHECKPOINT_PATH}')

print(f'Size: {CHECKPOINT_PATH.stat().st_size / 1024:.1f} KiB')

Checkpoint already exists: /workspace/users/yijin/boot_env/pjt_main/data/models/pretrain/stgcnpp/stgcnpp_ntu60_xsub_hrnet_j.pth
Size: 5716.9 KiB


In [3]:
# Checkpoint 구조 확인
import torch

sd = torch.load(str(CHECKPOINT_PATH), map_location='cpu', weights_only=False)
if 'state_dict' in sd:
    sd = sd['state_dict']

# Block별 채널 구성 요약
print(f'Total keys: {len(sd)}')
print(f'Total params: {sum(v.numel() for v in sd.values()):,}')
print()
print('Block architecture (from checkpoint):')
for blk in range(10):
    conv_w = sd.get(f'backbone.gcn.{blk}.gcn.conv.weight')
    A = sd.get(f'backbone.gcn.{blk}.gcn.A')
    if conv_w is not None:
        out_c = conv_w.shape[0] // 3
        in_c = conv_w.shape[1]
        has_down = f'backbone.gcn.{blk}.gcn.down.0.weight' in sd
        has_res = f'backbone.gcn.{blk}.residual.conv.weight' in sd
        print(f'  Block {blk}: in={in_c:3d} out={out_c:3d}  A={tuple(A.shape)}'
              f'  down={has_down}  residual={has_res}')
print()
print(f"cls_head: {tuple(sd['cls_head.fc_cls.weight'].shape)} (will be discarded)")

Total keys: 692
Total params: 1,405,674

Block architecture (from checkpoint):
  Block 0: in=  3 out= 64  A=(3, 17, 17)  down=True  residual=False
  Block 1: in= 64 out= 64  A=(3, 17, 17)  down=False  residual=False
  Block 2: in= 64 out= 64  A=(3, 17, 17)  down=False  residual=False
  Block 3: in= 64 out= 64  A=(3, 17, 17)  down=False  residual=False
  Block 4: in= 64 out=128  A=(3, 17, 17)  down=True  residual=True
  Block 5: in=128 out=128  A=(3, 17, 17)  down=False  residual=False
  Block 6: in=128 out=128  A=(3, 17, 17)  down=False  residual=False
  Block 7: in=128 out=256  A=(3, 17, 17)  down=True  residual=True
  Block 8: in=256 out=256  A=(3, 17, 17)  down=False  residual=False
  Block 9: in=256 out=256  A=(3, 17, 17)  down=False  residual=False

cls_head: (60, 256) (will be discarded)


## 3. Dance Reference 데이터 확인

fine-tuning 원본은 `data/reference_dances/*/reference.npy`이다.
모델 입력으로는 12개 DANCE_JOINTS를 선택하고, hip center 정규화 + shoulder width 스케일링을 적용한다.

In [4]:
import numpy as np

DANCE_DATA_DIR = PROJECT_ROOT / 'data' / 'reference_dances'
OUTPUT_DIR = PROJECT_ROOT / 'data' / 'models' / 'embedding'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('DANCE_DATA_DIR =', DANCE_DATA_DIR)
print('OUTPUT_DIR =', OUTPUT_DIR)
print()

total_frames = 0
print(f'{"name":<20s} {"shape":<20s} {"duration@30fps":>14s}')
print('-' * 56)
for ref_path in sorted(DANCE_DATA_DIR.glob('*/reference.npy')):
    arr = np.load(ref_path, allow_pickle=True)
    total_frames += arr.shape[0]
    print(f'{ref_path.parent.name:<20s} {str(arr.shape):<20s} {arr.shape[0]/30:.1f}s')
print('-' * 56)
print(f'{"TOTAL":<20s} {total_frames} frames{"":>8s} {total_frames/30:.1f}s')

DANCE_DATA_DIR = /workspace/users/yijin/boot_env/pjt_main/data/reference_dances
OUTPUT_DIR = /workspace/users/yijin/boot_env/pjt_main/data/models/embedding

name                 shape                duration@30fps
--------------------------------------------------------
404_dance            (568, 33, 3)         18.9s
beginner_wave        (1800, 33, 4)        60.0s
cheerup_dance        (724, 33, 4)         24.1s
freestyle_free       (3600, 33, 4)        120.0s
hiphop_move          (792, 33, 4)         26.4s
kpop_basic           (2700, 33, 4)        90.0s
--------------------------------------------------------
TOTAL                10184 frames         339.5s


## 4. Fine-Tuning 설정

`FINETUNE_CONFIGS`에 여러 모델 변형을 넣으면 순차적으로 학습한다.

- `name`: 저장할 모델 이름
- `embedding_dim`: 임베딩 차원 (64 또는 32)

스크립트 `scripts/finetune_pretrained_stgcn.py`가 각 config를 처리한다.

### Joint 매핑

ST-GCN++ pretrain은 COCO 17 joints를 사용한다. 우리 12 DANCE_JOINTS는 COCO의 5~16번에 해당한다.

```
COCO 17 joints          →  12 DANCE_JOINTS
5:  left_shoulder        →  0
6:  right_shoulder       →  1
7:  left_elbow           →  2
8:  right_elbow          →  3
9:  left_wrist           →  4
10: right_wrist          →  5
11: left_hip             →  6
12: right_hip            →  7
13: left_knee            →  8
14: right_knee           →  9
15: left_ankle           →  10
16: right_ankle          →  11
```

가중치 적응:
- `data_bn`: (3*17) → (2*12) — joint + channel 축소
- `gcn.A`: (3,17,17) → (3,12,12) — adjacency sub-selection
- `gcn.0.gcn.conv`: (192,3,1,1) → (192,2,1,1) — input channel 축소
- 나머지 Conv/BN: shape 동일 → 그대로 로드
- `cls_head`: 버림, `projection` head 새로 학습

In [5]:
# Fine-tuning settings.
# Defaults are smoke-run friendly. Increase epochs/steps for real training.
EPOCHS = 50
STEPS_PER_EPOCH = 60
VALIDATION_STEPS = 20
BATCH_SIZE = 64
PATIENCE = 7
LEARNING_RATE = 3e-4
POSITIVE_JITTER = 6
NEGATIVE_GAP = 600
NOISE_STD = 0.015
TRIPLET_MARGIN = 0.2

# Device: 'auto' (CUDA if available), 'cpu', or 'cuda'
# GPU 메모리 부족 시 'cpu'로 변경
DEVICE = 'cpu'

# Set ONLY to a list of names to train a subset.
ONLY = []

FINETUNE_CONFIGS = [
    {
        'name': 'stgcnpp_dance_e64',
        'embedding_dim': 64,
    },
    {
        'name': 'stgcnpp_dance_e32',
        'embedding_dim': 32,
    },
]

SELECTED_CONFIGS = [cfg for cfg in FINETUNE_CONFIGS if not ONLY or cfg['name'] in ONLY]
assert SELECTED_CONFIGS, 'No configs selected'

print('Checkpoint:', CHECKPOINT_PATH)
print('Output dir:', OUTPUT_DIR)
print('Device:', DEVICE)
print()
print('Selected configs:')
for cfg in SELECTED_CONFIGS:
    print(f"  - {cfg['name']}  embedding_dim={cfg['embedding_dim']}")

Checkpoint: /workspace/users/yijin/boot_env/pjt_main/data/models/pretrain/stgcnpp/stgcnpp_ntu60_xsub_hrnet_j.pth
Output dir: /workspace/users/yijin/boot_env/pjt_main/data/models/embedding
Device: cpu

Selected configs:
  - stgcnpp_dance_e64  embedding_dim=64
  - stgcnpp_dance_e32  embedding_dim=32


## 5. Fine-Tuning 실행

각 config는 `scripts/finetune_pretrained_stgcn.py`를 호출한다.

이 스크립트는:
1. PyTorch로 ST-GCN++ 모델을 빌드
2. PYSKL checkpoint에서 가중치 로드 (17→12 joint 적응)
3. classification head 대신 embedding projection head 추가
4. Triplet loss로 댄스 데이터에 fine-tune
5. PyTorch → ONNX → TFLite 변환
6. ScratchPoseSimilarity로 runtime smoke test

저장되는 것:
- `{name}.tflite` — 서비스 런타임 모델
- `{name}.pt` — PyTorch 가중치 (재학습/디버깅용)
- `{name}_meta.json` — 설정, 학습 이력, smoke metrics

In [6]:
FINETUNE_RESULTS = []

for index, cfg in enumerate(SELECTED_CONFIGS, start=1):
    print('=' * 100)
    print(f"[{index}/{len(SELECTED_CONFIGS)}] Fine-tuning {cfg['name']}")
    print('=' * 100)

    cmd = [
        sys.executable, str(PROJECT_ROOT / 'scripts' / 'finetune_pretrained_stgcn.py'),
        '--checkpoint', str(CHECKPOINT_PATH),
        '--data-dir', str(DANCE_DATA_DIR),
        '--model-name', cfg['name'],
        '--output-dir', str(OUTPUT_DIR),
        '--embedding-dim', str(cfg.get('embedding_dim', 64)),
        '--device', cfg.get('device', DEVICE),
        '--epochs', str(cfg.get('epochs', EPOCHS)),
        '--steps-per-epoch', str(cfg.get('steps_per_epoch', STEPS_PER_EPOCH)),
        '--validation-steps', str(cfg.get('validation_steps', VALIDATION_STEPS)),
        '--batch-size', str(cfg.get('batch_size', BATCH_SIZE)),
        '--patience', str(cfg.get('patience', PATIENCE)),
        '--learning-rate', str(cfg.get('learning_rate', LEARNING_RATE)),
        '--triplet-margin', str(cfg.get('triplet_margin', TRIPLET_MARGIN)),
        '--positive-jitter', str(cfg.get('positive_jitter', POSITIVE_JITTER)),
        '--negative-gap', str(cfg.get('negative_gap', NEGATIVE_GAP)),
        '--noise-std', str(cfg.get('noise_std', NOISE_STD)),
    ]
    if cfg.get('no_quantize', False):
        cmd += ['--no-quantize']

    started = time.time()
    print(' '.join(cmd))
    subprocess.check_call(cmd, cwd=PROJECT_ROOT)
    elapsed = time.time() - started

    meta_path = OUTPUT_DIR / f"{cfg['name']}_meta.json"
    meta = json.loads(meta_path.read_text(encoding='utf-8')) if meta_path.exists() else {}
    FINETUNE_RESULTS.append({
        'name': cfg['name'],
        'embedding_dim': cfg.get('embedding_dim', 64),
        'elapsed_sec': round(elapsed, 1),
        'training_summary': meta.get('training_summary', {}),
        'smoke_metrics': meta.get('smoke_metrics', {}),
        'runtime_smoke': meta.get('runtime_smoke', {}),
        'pt': str(OUTPUT_DIR / f"{cfg['name']}.pt"),
        'tflite': str(OUTPUT_DIR / f"{cfg['name']}.tflite"),
        'meta': str(meta_path),
        'service_command': f"python src/main.py -s embedding --embedding-model-name {cfg['name']}",
    })

print()
print('All selected fine-tuning runs finished.')
for result in FINETUNE_RESULTS:
    print(result)

[1/2] Fine-tuning stgcnpp_dance_e64
/workspace/users/yijin/boot_env/.venv/bin/python /workspace/users/yijin/boot_env/pjt_main/scripts/finetune_pretrained_stgcn.py --checkpoint /workspace/users/yijin/boot_env/pjt_main/data/models/pretrain/stgcnpp/stgcnpp_ntu60_xsub_hrnet_j.pth --data-dir /workspace/users/yijin/boot_env/pjt_main/data/reference_dances --model-name stgcnpp_dance_e64 --output-dir /workspace/users/yijin/boot_env/pjt_main/data/models/embedding --embedding-dim 64 --device cpu --epochs 50 --steps-per-epoch 60 --validation-steps 20 --batch-size 64 --patience 7 --learning-rate 0.0003 --triplet-margin 0.2 --positive-jitter 6 --negative-gap 600 --noise-std 0.015
[DEVICE] cpu
[DATA] 6 dance sequences:
  - 404_dance: 568 frames (568, 33, 3)
  - beginner_wave: 1800 frames (1800, 33, 4)
  - cheerup_dance: 724 frames (724, 33, 4)
  - freestyle_free: 3600 frames (3600, 33, 4)
  - hiphop_move: 792 frames (792, 33, 4)
  - kpop_basic: 2700 frames (2700, 33, 4)
[MODEL] ST-GCN++ Embedding: 1,

W0416 18:15:05.820000 2355424 torch/onnx/_internal/exporter/_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features
W0416 18:15:06.286000 2355424 torch/onnx/_internal/exporter/_registration.py:110] torchvision is not installed. Skipping torchvision::nms
W0416 18:15:06.287000 2355424 torch/onnx/_internal/exporter/_registration.py:110] torchvision is not installed. Skipping torchvision::roi_align
W0416 18:15:06.287000 2355424 torch/onnx/_internal/exporter/_registration.py:110] torchvision is not installed. Skipping torchvision::roi_pool


[torch.onnx] Obtain model graph for `STGCNPPEmbedding([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `STGCNPPEmbedding([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅


...

[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅


The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 17).
Failed to convert the model to the target version 17 using the ONNX C API. The model was not modified
Traceback (most recent call last):
  File "/workspace/users/yijin/boot_env/.venv/lib/python3.12/site-packages/onnxscript/version_converter/__init__.py", line 120, in call
    converted_proto = _c_api_utils.call_onnx_api(
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/workspace/users/yijin/boot_env/.venv/lib/python3.12/site-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
             ^^^^^^^^^^^
  File "/workspace/users/yijin/boot_env/.venv/lib/python3.12/site-packages/onnxscript/version_converter/__init__.py", line 115, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

[torch.onnx] Optimize the ONNX graph...
Applied 53 of general pattern rewrite rules.
[torch.onnx] Optimize the ONNX graph... ✅
[EXPORT] ONNX: /workspace/users/yijin/boot_env/pjt_main/data/models/embedding/stgcnpp_dance_e32.onnx (906.0 KiB)
[EXPORT] TFLite: /workspace/users/yijin/boot_env/pjt_main/data/models/embedding/stgcnpp_dance_e32.tflite (5351.2 KiB)


2026-04-16 18:44:19.655585: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:479] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-04-16 18:44:19.669275: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:10575] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-04-16 18:44:19.669304: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1442] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-16 18:44:19.680081: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-04-16 18:44:20.552183: W tensorflow/compiler/tf

[VERIFY] input=[1, 30, 12, 2] output=[1, 32] embedding_norm=1.0000
[SMOKE] same=1.0000 cross=0.0000 finite=True
[SAVE] Metadata: /workspace/users/yijin/boot_env/pjt_main/data/models/embedding/stgcnpp_dance_e32_meta.json

Run service with:
  python src/main.py -s embedding --embedding-model-name stgcnpp_dance_e32
  python src/main.py -s embedding --embedding-model-path /workspace/users/yijin/boot_env/pjt_main/data/models/embedding/stgcnpp_dance_e32.tflite

All selected fine-tuning runs finished.
{'name': 'stgcnpp_dance_e64', 'embedding_dim': 64, 'elapsed_sec': 3036.4, 'training_summary': {'epochs_ran': 26, 'best_epoch': 19, 'best_val_loss': 0.015250462584663182}, 'smoke_metrics': {'same_cosine_mean': 0.9691258366219699, 'different_cosine_mean': 0.06864487251732498, 'margin_mean': 0.900480964104645}, 'runtime_smoke': {'same': 1.0, 'cross': None}, 'pt': '/workspace/users/yijin/boot_env/pjt_main/data/models/embedding/stgcnpp_dance_e64.pt', 'tflite': '/workspace/users/yijin/boot_env/pjt_mai

## 5.1 결과 파일과 Metadata 확인

각 모델의 `training_summary`에는 best epoch, best validation loss가 들어간다.
`smoke_metrics`는 같은 춤의 가까운 window와 다른 춤 window의 cosine 차이를 보여준다.

In [6]:
if 'FINETUNE_RESULTS' not in globals() or not FINETUNE_RESULTS:
    FINETUNE_RESULTS = []
    for cfg in SELECTED_CONFIGS:
        meta_path = OUTPUT_DIR / f"{cfg['name']}_meta.json"
        if not meta_path.exists():
            continue
        meta = json.loads(meta_path.read_text(encoding='utf-8'))
        FINETUNE_RESULTS.append({
            'name': cfg['name'],
            'embedding_dim': cfg.get('embedding_dim', 64),
            'training_summary': meta.get('training_summary', {}),
            'smoke_metrics': meta.get('smoke_metrics', {}),
            'runtime_smoke': meta.get('runtime_smoke', {}),
            'pt': str(OUTPUT_DIR / f"{cfg['name']}.pt"),
            'tflite': str(OUTPUT_DIR / f"{cfg['name']}.tflite"),
            'meta': str(meta_path),
            'service_command': f"python src/main.py -s embedding --embedding-model-name {cfg['name']}",
        })

for result in FINETUNE_RESULTS:
    print('-' * 100)
    print(f"{result['name']}  (embedding_dim={result['embedding_dim']})")
    print(f"  training: {result.get('training_summary', {})}")
    print(f"  smoke:    {result.get('smoke_metrics', {})}")
    print(f"  runtime:  {result.get('runtime_smoke', {})}")
    for key in ('pt', 'tflite', 'meta'):
        path = Path(result[key])
        size = f'{path.stat().st_size / 1024:.1f} KiB' if path.exists() else 'missing'
        print(f'  {key}: {path.name} ({size})')
    print(f"  run: {result['service_command']}")

----------------------------------------------------------------------------------------------------
stgcnpp_dance_e64  (embedding_dim=64)
  training: {'epochs_ran': 26, 'best_epoch': 19, 'best_val_loss': 0.015250462584663182}
  smoke:    {'same_cosine_mean': 0.9691258366219699, 'different_cosine_mean': 0.06864487251732498, 'margin_mean': 0.900480964104645}
  runtime:  {'same': 1.0, 'cross': None}
  pt: stgcnpp_dance_e64.pt (5709.8 KiB)
  tflite: stgcnpp_dance_e64.tflite (5383.5 KiB)
  meta: stgcnpp_dance_e64_meta.json (5.4 KiB)
  run: python src/main.py -s embedding --embedding-model-name stgcnpp_dance_e64
----------------------------------------------------------------------------------------------------
stgcnpp_dance_e32  (embedding_dim=32)
  training: {'epochs_ran': 15, 'best_epoch': 8, 'best_val_loss': 0.024668092257343233}
  smoke:    {'same_cosine_mean': 0.9570008800365031, 'different_cosine_mean': 0.10618951999640558, 'margin_mean': 0.8508113600400975}
  runtime:  {'same': 1.0,

## 6. 기존 Embedding 모델과 비교

기존 MPOSE2021 pretrain → fine-tune 모델과 ST-GCN++ pretrain → fine-tune 모델의
smoke metrics를 비교한다.

In [7]:
# Collect all embedding models for comparison.
all_models = []
for meta_path in sorted(OUTPUT_DIR.glob('*_meta.json')):
    meta = json.loads(meta_path.read_text(encoding='utf-8'))
    name = meta.get('model_name', meta_path.stem.replace('_meta', ''))
    sm = meta.get('smoke_metrics', {})
    ts = meta.get('training_summary', {})
    model_type = meta.get('model_type', meta.get('encoder_config', {}).get('architecture', '?'))
    tflite_path = meta_path.parent / f"{name}.tflite"
    tflite_size = f"{tflite_path.stat().st_size / 1024:.0f}" if tflite_path.exists() else '-'
    all_models.append({
        'name': name, 'type': model_type,
        'same': sm.get('same_cosine_mean', 0),
        'diff': sm.get('different_cosine_mean', 0),
        'margin': sm.get('margin_mean', 0),
        'best_epoch': ts.get('best_epoch', '-'),
        'tflite_kib': tflite_size,
    })

# Sort by margin descending.
all_models.sort(key=lambda x: x['margin'], reverse=True)

print(f'{"model_name":<40s} {"type":<20s} {"same":>6s} {"diff":>6s} {"margin":>7s} {"epoch":>6s} {"KiB":>5s}')
print('-' * 90)
for m in all_models:
    print(f"{m['name']:<40s} {m['type']:<20s} {m['same']:>6.3f} {m['diff']:>6.3f} {m['margin']:>7.3f} {str(m['best_epoch']):>6s} {m['tflite_kib']:>5s}")

model_name                               type                   same   diff  margin  epoch   KiB
------------------------------------------------------------------------------------------
embedding_tcn_e64_triplet                ?                     0.957 -0.030   0.987     10    99
embedding_gcn_e32_triplet                ?                     0.947  0.009   0.938     21   466
embedding_gcn_e64_triplet                ?                     0.951  0.050   0.901     16   469
stgcnpp_dance_e64                        stgcnpp_pretrained    0.969  0.069   0.900     19  5383
stgcnpp_dance_e32                        stgcnpp_pretrained    0.957  0.106   0.851      8  5351
embedding_tcn_e32_triplet                ?                     0.944  0.161   0.783      5    97


## 7. 유사도와 스켈레톤 애니메이션 확인

fine-tuned TFLite encoder가 실제 댄스 window들을 어떻게 embedding하는지 확인한다.

- `anchor`: 기준 window
- `positive`: 같은 춤에서 가까운 시점 → cosine similarity 높게 기대
- `negative`: 다른 춤 → cosine similarity 낮게 기대

In [8]:
import tensorflow as tf
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

from pose.landmark_utils import DANCE_JOINTS
from scoring.scratch_features import build_pose_window

# Pick the best ST-GCN++ model or set manually.
VIS_MODEL_NAME = None  # Set to override, e.g. 'stgcnpp_dance_e64'

if VIS_MODEL_NAME is None:
    stgcnpp_results = [r for r in FINETUNE_RESULTS if r.get('name', '').startswith('stgcnpp')]
    if stgcnpp_results:
        VIS_MODEL_NAME = stgcnpp_results[0]['name']
    else:
        VIS_MODEL_NAME = FINETUNE_RESULTS[0]['name'] if FINETUNE_RESULTS else None

assert VIS_MODEL_NAME, 'No model found for visualization'

model_path = OUTPUT_DIR / f'{VIS_MODEL_NAME}.tflite'
meta_path = OUTPUT_DIR / f'{VIS_MODEL_NAME}_meta.json'
meta = json.loads(meta_path.read_text(encoding='utf-8'))
config = meta.get('encoder_config', {}) or meta.get('config', {}) or {}
input_shape = meta.get('input_shape') or [1, 30, 12, 2]
SEQ_LEN = int(config.get('sequence_length', input_shape[1]))
FEAT_DIMS = int(config.get('feature_dims', input_shape[-1]))

# Samples: (label, dance_name, end_frame)
requested_samples = [
    ('anchor',   'cheerup_dance', 300),
    ('positive', 'cheerup_dance', 308),
    ('negative', 'hiphop_move',   300),
]
available_dances = sorted(p.parent.name for p in (PROJECT_ROOT / 'data' / 'reference_dances').glob('*/reference.npy'))

def existing_dance(name, fallback_index=0):
    if name in available_dances:
        return name
    return available_dances[min(fallback_index, len(available_dances) - 1)]

SAMPLES = [
    (requested_samples[0][0], existing_dance(requested_samples[0][1], 0), requested_samples[0][2]),
    (requested_samples[1][0], existing_dance(requested_samples[1][1], 0), requested_samples[1][2]),
    (requested_samples[2][0], existing_dance(requested_samples[2][1], 1), requested_samples[2][2]),
]

DANCE_EDGES = [(0,1),(6,7),(0,6),(1,7),(0,2),(2,4),(1,3),(3,5),(6,8),(8,10),(7,9),(9,11)]
COLOR_MAP = {'anchor': 'tab:gray', 'positive': 'tab:green', 'negative': 'tab:red'}

# Load windows
ref_dir = PROJECT_ROOT / 'data' / 'reference_dances'
entries = []
for label, dance, end_f in SAMPLES:
    seq = np.load(ref_dir / dance / 'reference.npy', allow_pickle=True).astype(np.float32)
    end_f = max(SEQ_LEN - 1, min(end_f, len(seq) - 1))
    start_f = end_f - SEQ_LEN + 1
    window = build_pose_window(seq, end_f, SEQ_LEN, target_joints=DANCE_JOINTS, feature_dims=FEAT_DIMS)
    raw = seq[start_f:end_f+1][:, DANCE_JOINTS, :2].copy()
    raw = np.nan_to_num(raw, nan=0.0, posinf=0.0, neginf=0.0)
    entries.append({'label': label, 'dance': dance, 'end': end_f, 'window': window, 'raw_window': raw})

# Embed with TFLite
interp = tf.lite.Interpreter(model_path=str(model_path))
interp.allocate_tensors()
in_det, out_det = interp.get_input_details()[0], interp.get_output_details()[0]

def embed(w):
    t = w[None, ...].astype(np.float32)
    interp.set_tensor(in_det['index'], t)
    interp.invoke()
    e = interp.get_tensor(out_det['index']).reshape(-1).astype(np.float32)
    n = np.linalg.norm(e)
    return e / n if n > 1e-8 else e

def cosine(a, b):
    a, b = np.nan_to_num(np.asarray(a, np.float32)), np.nan_to_num(np.asarray(b, np.float32))
    na, nb = np.linalg.norm(a), np.linalg.norm(b)
    return float(np.dot(a, b) / (na * nb)) if na > 1e-8 and nb > 1e-8 else 0.0

for e in entries:
    e['emb'] = embed(e['window'])

# Print info
print(f'[MODEL] {VIS_MODEL_NAME}')
print(f'input_shape = {list(in_det["shape"])}  output_shape = {list(out_det["shape"])}')
print(f'training_summary = {meta.get("training_summary", {})}')
print()
print('samples:')
for e in entries:
    print(f"  {e['label']:>8s}: {e['dance']} end={e['end']}")
print()
print('pairwise cosine similarity:')
labels = [e['label'] for e in entries]
print(' ' * 12 + ''.join(f'{l:>11s}' for l in labels))
for ei in entries:
    row = f'{ei["label"]:>10s}: '
    for ej in entries:
        row += f'{cosine(ei["emb"], ej["emb"]):+10.4f} '
    print(row)

# --- axis limits (same as embedding_finetuning.ipynb) ---
def axis_limits_raw(raw_windows, pad=0.03):
    arr = np.stack(raw_windows, axis=0)
    x, y = arr[..., 0], arr[..., 1]
    valid = np.isfinite(x) & np.isfinite(y) & (x > 0.01) & (y > 0.01)
    if not np.any(valid):
        return (0.0, 1.0), (0.0, 1.0)
    return (
        (float(x[valid].min()) - pad, float(x[valid].max()) + pad),
        (float(y[valid].min()) - pad, float(y[valid].max()) + pad),
    )

def draw_skeleton_raw(ax, pts, color):
    pts = np.nan_to_num(pts, nan=0.0, posinf=0.0, neginf=0.0)
    xs, ys = pts[:, 0], -pts[:, 1]
    scatter = ax.scatter(xs, ys, c=color, s=30, zorder=3)
    lines = []
    for a, b in DANCE_EDGES:
        line = ax.plot([xs[a], xs[b]], [ys[a], ys[b]], color=color, lw=1.8, alpha=0.85)[0]
        lines.append(line)
    return scatter, lines

xlim, (y_lo, y_hi) = axis_limits_raw([e['raw_window'] for e in entries])
ylim = (-y_hi, -y_lo)
anchor_emb = entries[0]['emb']

fig, axes = plt.subplots(1, len(entries), figsize=(4 * len(entries), 4.5))
if len(entries) == 1:
    axes = [axes]

artists_by_ax = []
for ax, e in zip(axes, entries):
    sim = cosine(anchor_emb, e['emb'])
    ax.set_title(
        f"{e['label']}\n{e['dance']} (end={e['end']})\n"
        f"cos(anchor, .) = {sim:+.3f}",
        fontsize=10,
    )
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_aspect('equal')
    ax.set_xticks([])
    ax.set_yticks([])
    artists_by_ax.append([])

frame_text = fig.text(0.5, 0.02, '', ha='center')

def update(frame_idx):
    for old in artists_by_ax:
        for artist in old:
            artist.remove()
        old.clear()
    for ax, e, old in zip(axes, entries, artists_by_ax):
        color = COLOR_MAP.get(e['label'], 'tab:blue')
        scatter, lines = draw_skeleton_raw(ax, e['raw_window'][frame_idx], color)
        old.extend([scatter, *lines])
    frame_text.set_text(f'frame {frame_idx + 1} / {SEQ_LEN}')
    return [frame_text] + [a for old in artists_by_ax for a in old]

anim = FuncAnimation(fig, update, frames=SEQ_LEN, interval=1000 / 30, blit=False)
plt.close(fig)
HTML(anim.to_jshtml())

2026-04-17 08:31:52.612627: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:479] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-04-17 08:31:52.628396: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:10575] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-04-17 08:31:52.628426: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1442] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-17 08:31:52.640057: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-04-17 08:31:53.267888: W tensorflow/compiler/tf

[MODEL] stgcnpp_dance_e64
input_shape = [1, 30, 12, 2]  output_shape = [1, 64]
training_summary = {'epochs_ran': 26, 'best_epoch': 19, 'best_val_loss': 0.015250462584663182}

samples:
    anchor: cheerup_dance end=300
  positive: cheerup_dance end=308
  negative: hiphop_move end=300

pairwise cosine similarity:
                 anchor   positive   negative
    anchor:    +1.0000    +0.9959    -0.1768 
  positive:    +0.9959    +1.0000    -0.1799 
  negative:    -0.1768    -0.1799    +1.0000 


## 8. Runtime Smoke Test

서비스와 같은 `ScratchPoseSimilarity.compute()` 경로로 fine-tuned TFLite 모델을 호출한다.
이 테스트가 finite score를 내면 `src/main.py -s embedding`에서도 같은 방식으로 동작한다.

In [9]:
from scoring.scratch_similarity import ScratchPoseSimilarity


def runtime_smoke(model_name):
    tflite_p = OUTPUT_DIR / f'{model_name}.tflite'
    meta_p = OUTPUT_DIR / f'{model_name}_meta.json'
    assert tflite_p.exists(), f'Missing: {tflite_p}'
    m = json.loads(meta_p.read_text('utf-8')) if meta_p.exists() else {}
    cfg = m.get('encoder_config', {}) or m.get('config', {})
    sl = int(cfg.get('sequence_length', 30))
    fd = int(cfg.get('feature_dims', 2))

    ref = np.load(DANCE_DATA_DIR / 'beginner_wave' / 'reference.npy', allow_pickle=True)
    other = np.load(DANCE_DATA_DIR / 'hiphop_move' / 'reference.npy', allow_pickle=True)
    comp = ScratchPoseSimilarity(str(tflite_p), sequence_length=sl, feature_dims=fd, input_layout='BTJC')

    same = None
    for idx in range(sl):
        same = comp.compute(ref[idx], ref, idx)
    comp.reset()
    cross = None
    for idx in range(sl):
        cross = comp.compute(other[idx], ref, idx)

    ok = same is not None and cross is not None and np.isfinite(same) and np.isfinite(cross)
    print(f'{model_name:40s} same={same}  cross={cross}  finite={ok}')


models_to_check = [r['name'] for r in FINETUNE_RESULTS] if FINETUNE_RESULTS else []
if not models_to_check:
    models_to_check = [p.stem for p in sorted(OUTPUT_DIR.glob('stgcnpp_*.tflite'))]

for name in models_to_check:
    runtime_smoke(name)

stgcnpp_dance_e64                        same=1.0  cross=0.07157672941684723  finite=True
stgcnpp_dance_e32                        same=1.0  cross=0.0  finite=True


## 9. 서비스 실행 명령

fine-tuned `.tflite` 모델은 `embedding` score method에서 사용한다.
내부적으로는 scratch runtime과 같은 window → embedding → cosine 계약을 쓰지만,
모델은 **PYSKL ST-GCN++ (NTU-60 pretrain)** 후 댄스 reference로 fine-tuning된 encoder다.

In [ ]:
BEST_MODEL_NAME = FINETUNE_RESULTS[0]['name'] if FINETUNE_RESULTS else None
if BEST_MODEL_NAME:
    print('Run service with:')
    print(f'  python src/main.py -s embedding --embedding-model-name {BEST_MODEL_NAME}')
    print()
    print('Or with direct path:')
    print(f'  python src/main.py -s embedding --embedding-model-path data/models/embedding/{BEST_MODEL_NAME}.tflite')
else:
    print('No fine-tuned models found yet.')